# DenseNet121 : Images - Iteration #5_1 : Without Data Augementation - LR Optimization 

img_rows = 224  <br>
img_cols = 224 <br>
batch_size =  64 <br>
epochs = 100 <br>
Train set size = 67933 <br>
Valid set size = 16983 <br>
Without Data augmentation <br>
Freeze layers of the Base model layers  <br>
Optimize LR with LearningRateScheduler <br>
Start with small LR: 0,00001 then increase the value evry  epoch until 2.54 <br>
then elect a learning rate that achieves the lowest loss <br>

**<u>Results</u>**

Accuracy : 0.?? <br>
F1 score weighted : 0.?? <br>

**Import packages**

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import cv2
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from tqdm.notebook import tqdm_notebook
import re
tqdm_notebook.pandas()
import tensorflow as tf
#from tensorflow import keras
from tensorflow.keras.models import load_model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.efficientnet import EfficientNetB7
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2
from tensorflow.keras.applications.densenet import DenseNet121
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import f1_score , recall_score, accuracy_score, precision_score, confusion_matrix
from keras.utils import np_utils
from sklearn.model_selection import train_test_split
from sklearn import metrics
import itertools
import pickle
from joblib import dump, load

# Iteration #5_1 - DenseNet121

***Read DataFrame from saved pickele file - image Train set***

In [ ]:
#Load df of images already resized on 256 x 256 
df_train_im = pd.read_pickle("./saves/df_save/df_train_combined_19102021.pkl")

#Load df of images with org size 500 x 500
#df_train_im = pd.read_pickle("./saves/df_save/df_train_combined_ORG_im_size_23112021.pkl")

In [ ]:
df_train_im.head()

In [ ]:
display(df_train_im.shape)

***Create Dataframe containing  product classes and their associated labels***

In [ ]:
#Dictionary of prdtypecode and their corresponding categories
dict_prdtypecode = {"prdtypecode" : [50, 2705, 2522, 2582, 1560, 1281, 1920, 1280, 1140, 1300, 2060, 2583,
                                     60, 1320, 2280, 1302, 2220, 40, 2905, 2585, 1940, 1160, 1301, 10, 1180,
                                     2403, 2462],                 
            
                    "Label" : ["video games accessories", "books", "stationery", "kitchen and garden", 
                               "interior furniture and bedding", "board games", "interior accessories",
                               "toys for children","goodies", "remote controlled models", "decoration interior",
                               "piscine spa","games and consoles", "early childhood", "magazines", "kids toys",
                               "supplies for domestic animals", "imported video games", "online distribution of video games",
                               "gardening and DIY","Food","playing cards", "accessories children", "adult books",
                               "figurines_wargames", "children books",
                                "games"]
                   }
df_class = pd.DataFrame(data=dict_prdtypecode)
df_class

In [ ]:
# Crate DF containing prdtypecode , categories and corresponding class used by model
df_class = df_class.sort_values(by = 'prdtypecode', ascending = True)
df_class['target_prdtypecode'] = [i for i in range(27)]
df_class

***Change the 27 product codes to 0 to 26***

In [ ]:
df_train_im.replace({'prdtypecode':{10:0,
                                    40:1,
                                    50:2,
                                    60:3,
                                    1140:4,
                                    1160:5,
                                    1180:6,
                                    1280:7,
                                    1281:8,
                                    1300:9                                    
                                    }}, inplace = True)

df_train_im.replace({'prdtypecode':{1301:10,
                                    1302:11,
                                    1320:12,
                                    1560:13,
                                    1920:14,
                                    1940:15,
                                    2060:16,
                                    2220:17,
                                    2280:18,
                                    2403:19,
                                    2462:20,
                                    2522:21,
                                    2582:22,
                                    2583:23,
                                    2585:24,
                                    2705:25,
                                    2905:26
                                    }}, inplace = True)

In [ ]:
display(df_train_im["prdtypecode"].unique())

 ***Convert product code to string type***

In [ ]:
df_train_im["prdtypecode"] = df_train_im["prdtypecode"].astype(str)

In [ ]:
unique, counts = np.unique(df_train_im["prdtypecode"], return_counts=True)
dict(zip(unique, counts))

 ***Split Train Set***

In [ ]:
# Split Train set 
#X_train_im, X_test_im = train_test_split(df_train_im, train_size=0.8, random_state=1234)

In [ ]:
#display(X_train_im.shape)
#display(X_test_im.shape)

***Data Generator***

In [ ]:
img_rows = 224
img_cols = 224
batch_size = 64
epochs = 100

# Directory of images already resized on 256 x 256 
images_dir_train = './data/images/all/image_train_resized/'
#images_dir_train = 'E:/WorkSpace_DataScience_E/Rakuten_Projet_Local/My_WS/data/images/all/image_train_resized/'

# Directory of images org resize 500 x 500
#images_dir_train = './data/images/all/image_train/'
#images_dir_train = 'E:/WorkSpace_DataScience_E/Rakuten_Projet_Local/My_WS/data/images/all/image_train/'

In [ ]:
%%time
#Générateur de données
img_gen = ImageDataGenerator(rescale = 1/255, 
                             validation_split = 0.2)

In [ ]:
%%time

#Itérateur 
train_generator = img_gen.flow_from_dataframe(dataframe = df_train_im,
                                              directory = images_dir_train,
                                              as_ext = True,
                                              x_col = 'imagePath',
                                              y_col = 'prdtypecode',  #target data                                            
                                              class_mode = 'sparse',                                             
                                              target_size = (img_rows , img_cols), # default 256 x 256                                             
                                              batch_size = batch_size,                                             
                                              shuffle = True, # défault
                                              subset = 'training'
                                             )

In [ ]:
%%time
# validation_split the validation batches ca be retrieved by specifying the subset as validation.
valid_generator = img_gen.flow_from_dataframe(dataframe = df_train_im, 
                                              directory = images_dir_train,
                                              as_ext = True,
                                              x_col = 'imagePath',
                                              y_col = 'prdtypecode',  #target data                                            
                                              class_mode = 'sparse',                                        
                                              target_size = (img_rows , img_cols), # default 256 x 256                                         
                                              batch_size = batch_size, 
                                              shuffle = False,
                                              subset  = 'validation'
                                             )

 ***DenseNet121 Implementation***

In [ ]:
%%time
# Chargement du modèle DenseNet121
denseNet121 = DenseNet121(weights='imagenet', include_top = False, input_shape=(img_rows,img_rows,3))

# Bloquage du blackbone
for layer in denseNet121.layers:
    layer.trainable = False

model = Sequential()
model.add(denseNet121)
# Ajout des couche de classification as VGG16 efficientNet , ResNet different!
model.add(GlobalAveragePooling2D()) 
model.add(Dense(units = 1024,activation='relu'))

model.add(Dropout(rate = 0.2))
model.add(Dense(units = 512, activation='relu'))

model.add(Dropout(rate = 0.2))
model.add(Dense(units = 27, activation='softmax'))
model.summary()

 ***Compilation***

In [ ]:
%%time
# Compilation
model.compile(loss='sparse_categorical_crossentropy',  # fonction de perte
              optimizer= Adam(learning_rate= 1e-5) ,  # algorithme de descente de gradient
              metrics=['accuracy'])                    # métrique d'évaluation

***Callbacks***

In [ ]:
def increasinglrUpdate(epoch, learning_rate):
    if epoch ==0:
        print("[INFO] : Starting : epoch: ", epoch , "learning_rate : ", learning_rate)
        return learning_rate
    else:
        lr_updated = 1e-5 * 100 ** (epoch / 37)
        print("[INFO] : Learning rate updated from:", learning_rate, 'to: ',  lr_updated)
        return lr_updated

In [ ]:
%%time
path = './saves/DenseNet121/Iteration_5_1/'
file_name = 'checkpoint_DenseNet121_10012022_100Epochs.h5'

#Sauvegarder les meilleurs poids du modèle au cours de l'entraînement :
checkpoint = ModelCheckpoint(filepath = path + file_name , 
                             monitor ='val_accuracy',                            
                             save_best_only = True, # (pour que le meilleur modèle ne soit pas écrasé)
                             save_weights_only = False,
                             mode ='max', #(permet de préciser si la métrique doit croître ou décroitre : ici on choisit 'min'
                                           #car la métrique est une perte à minimiser)
                             save_freq='epoch',
                             verbose = 1
                             )
#Arrêter l'entraînement si le modèle n'évolue plus (très pratique pour ne pas gérer le
#nombre d'epoch) :
early = EarlyStopping(monitor='val_loss',
                      min_delta = 0.01, # si au bout de 5 epochs la fonction de perte ne varie pas de 1%, 
                                         # que ce soit à la hausse ou à la baisse, on arrête
                      patience = 5, #(nombre d'epochs à attendre avant d'arrêter l'entraînement
                      restore_best_weights = True,                     
                      mode = 'min',
                      verbose = 1,
                      )
#ReduceLROnPlateau. Lorsque qu'une métrique ne s'améliore plus d'une epoch à l'autre, 
#on peut diminuer le taux d'apprentissage (*Learning Rate*) pour y remédier. 
#Pour définir les critères et les conditions de réduction de taux d'apprentissage, 
#on peut utiliser les paramètres suivants : 

lr_plateau = ReduceLROnPlateau(monitor = 'val_loss',
                               patience = 3,  #si val_loss stagne sur 3 epochs consécutives selon la valeur epsilon                            
                               mode='min',
                               min_delta = 0.01, # Si val loss n'a pas varié de 1 % on applique 
                               factor = 0.1,  # On réduit le learning rate d'un facteur 0.1 ( On divise le LR par 10)
                               cooldown = 4, # On attend 4 epochs avant de réitérer.
                                             #Nombre d'epochs de 'pause' entre deux mesures de suivis du *learning rate*.
                               verbose=1
                               )


lrScheduler = LearningRateScheduler(schedule = increasinglrUpdate, verbose = 1)
#lrScheduler = LearningRateScheduler(lambda epoch: 1e-3 * 10 ** (epoch / 37)
#We’ll train the model for 100 epochs to test 100 different loss/learning rate combinations.
#Here’s the range for the learning rate values: 1e-5 * 100 ** (0 / 37) and 1e-5 * 100 ** (100 / 37)
# (1e-05, 2.543345761304648)

 ***Fit - train_generator***

In [ ]:
step_size_Train = train_generator.n//train_generator.batch_size
step_size_Valid = valid_generator.n//valid_generator.batch_size
print('step_size_Train : ' , step_size_Train)
print('step_size_Valid  : ' , step_size_Valid)

In [ ]:
%%time
# Fit Train generator
history = model.fit(train_generator,                    
                    epochs = epochs,                   
                    steps_per_epoch = step_size_Train,
                    validation_data = valid_generator,
                    validation_steps = step_size_Valid,
                    #callbacks=[checkpoint, early ,lr_plateau]
                    callbacks=[lrScheduler] 
                   )

***Save History results***

In [ ]:
# Save History results
path = './saves/DenseNet121/Iteration_5_1/'
filename = 'history_DenseNet121_all_train_10012022_100Epochs'

In [ ]:
#convert the history.history dict to a pandas DataFrame:     
hist_df = pd.DataFrame(history.history) 

# history to json:  
hist_json_file = path + filename + '.json'
with open(hist_json_file, mode='w') as f:
    hist_df.to_json(f)

# history to csv: 
hist_csv_file = path + filename +'.csv'
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)

# history to pickle file
hist_df.to_pickle(path + filename +'.pkl')

***Display Model loss , accuracy per epoch***

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(121)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss by epoch')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'valid'], loc='right')

plt.subplot(122)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model acc by epoch')
plt.ylabel('acc')
plt.xlabel('epoch')
plt.legend(['train', 'valid'], loc='right')
plt.show()

In [ ]:
plt.plot(
    np.arange(1, 101), 
    history.history['lr'], 
    label='Learning rate', color='#000', lw=3, linestyle='--')

plt.plot(
    np.arange(1, 101), 
    history.history['loss'], 
    label='Loss', lw=3
)

In [ ]:
from matplotlib import rcParams
rcParams['figure.figsize'] = (18, 8)
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False 

plt.plot(
    np.arange(1, 101), 
    history.history['loss'], 
    label='Loss', lw=3
)
plt.plot(
    np.arange(1, 101), 
    history.history['accuracy'], 
    label='Accuracy', lw=3
)
plt.plot(
    np.arange(1, 101), 
    history.history['lr'], 
    label='Learning rate', color='#000', lw=3, linestyle='--'
)
plt.title('Evaluation metrics', size=20)
plt.xlabel('Epoch', size=14)
plt.legend();

In [ ]:
#1e-5 * 10 ** (epoch / 37)
learning_rates = 1e-5 * (10 ** (np.arange(100) / 37))
plt.semilogx(
    learning_rates, 
    history.history['loss'], 
    lw=3, color='#000'
)
plt.title('Learning rate vs. loss', size=20)
plt.xlabel('Learning rate', size=14)
plt.ylabel('Loss', size=14);

In [ ]:
plt.plot(
    np.arange(1, 101), 
    history.history['loss'], 
    label='Loss', lw=3
)
plt.plot(
    np.arange(1, 101), 
    history.history['accuracy'], 
    label='Accuracy', lw=3
)
plt.title('Accuracy vs. Loss per epoch', size=20)
plt.xlabel('Epoch', size=14)
plt.legend()

 ***Prediction - test_generator***

In [ ]:
%%time
#Prediction - Valid set
y_pred_proba = model.predict(valid_generator)

# l'argmax pour obtenir les classes prédites
y_pred_class = np.argmax(y_pred_proba,axis = 1).astype(int)

# To get classes from test generator
y_true = valid_generator.classes

 ***Evaluation - test_generator***

In [ ]:
%%time
#Get the accuracy score
valid_score = model.evaluate(valid_generator)
print("[INFO] Model metrics names:", model.metrics_names)
print("[INFO] Accuracy: {:.2f}%".format(valid_score[1] * 100)) 
print("[INFO] Loss: ",valid_score[0])

In [ ]:
f1_macro = f1_score(y_true, y_pred_class, average='macro')
print("[INFO] f1 score macro average: ",f1_macro)

f1_micro = f1_score(y_true, y_pred_class, average='micro')
print("[INFO] f1 score micro average: ",f1_micro)

f1_weighted = f1_score(y_true, y_pred_class, average='weighted')
print("[INFO] f1 score weighted average: ",f1_weighted)


In [ ]:
#Confusion matrix
matrix = confusion_matrix(y_true, y_pred_class)
fig , ax = plt.subplots(figsize = (24,20))
ax.matshow(matrix, cmap = plt.cm.Oranges , alpha = 0.3)
for i in range(matrix.shape[0]):
    for j in range (matrix.shape[1]):
        ax.text (x = j , y= i, s= matrix[i,j], va = 'center', ha = 'center', size = 'xx-large')
        
plt.xlabel('predicted', fontsize = 18)
plt.ylabel('Acutals', fontsize = 18)
plt.title('Confusion Matrix', fontsize = 18)
plt.show()
#print(matrix)

In [ ]:
print(metrics.classification_report(y_true, y_pred_class))

 ***Predict input image using trained model***

In [ ]:
# Predict With Pre Trained

# Load an image
im_id = 104
img_path ='./data/images/all/image_train_resized/'+ df_train_im['imagePath'][im_id]

img = image.load_img(img_path,target_size=(224, 224))
plt.figure(figsize=(6,4))
plt.imshow(img)
plt.title("class: " + str(df_train_im['prdtypecode'][im_id]))
plt.xticks([])
plt.yticks([])
#plt.axis('off')
plt.show();

In [ ]:
im = cv2.resize(cv2.imread(img_path), (224, 224))
im = np.expand_dims(im, axis=0)

In [ ]:
out = model.predict(im)
print("------------------Predicted Class and Label ------------------------------")
print("target class(used by model) :", np.argmax(out))
print("\n"  ,df_class[df_class['target_prdtypecode'] == np.argmax(out)][['prdtypecode', 'Label']].to_string(index=False)) 

prd_code = df_train_im['prdtypecode'][im_id]

print("\n\n------------------Real Class and Label ------------------------------")
print("target class(used by model): " ,  df_class[df_class['target_prdtypecode'] == int(prd_code)]['target_prdtypecode'].to_string(index=False))
print("\n"  ,df_class[df_class['target_prdtypecode'] == int(prd_code)][['prdtypecode', 'Label']].to_string(index=False)) 


 


 ***Save DenseNet121 Model***

In [ ]:
#Enregistrement du modèle
model.save("./saves/DenseNet121/Iteration_5_1/Model_DenseNet121_All_train_data_10012022_100Epochs.hdf5")
#The SavedModel and HDF5 file contains:
#the model's configuration (topology)
#the model's weights
#the model's optimizer's state (if any)

 ***Load DenseNet121 Model***

In [ ]:
#Rechargement du modèle :
model = load_model('./saves/DenseNet121/Iteration_5_1/Model_DenseNet121_All_train_data_10012022_100Epochs.hdf5')